# Anomaly detection and Time Series

# ============================================================
# QUESTION 1
# What is Anomaly Detection?
# ============================================================

# Anomaly Detection identifies data points that significantly differ
# from normal patterns.

# Types:

# 1. Point Anomaly:
# A single data point is abnormal.
# Example: A credit card transaction of ₹5,00,000 when normal transactions
# are around ₹1,000.

# 2. Contextual Anomaly:
# A data point is abnormal only in a particular context.
# Example: Electricity usage of 500 kWh may be normal in summer but abnormal
# during a mild-weather night.

# 3. Collective Anomaly:
# A group/sequence of data points is abnormal together.
# Example: A sudden sequence of unusual network requests indicating an attack.


# ============================================================
# QUESTION 2
# Isolation Forest vs DBSCAN vs LOF
# ============================================================

# Isolation Forest:
# - Randomly splits data to isolate unusual points.
# - Anomalies are isolated with fewer splits.
# - Good for large, high-dimensional numerical datasets.
# - Suitable for real-time anomaly detection.

# DBSCAN:
# - Groups points based on density.
# - Points outside dense regions are treated as noise.
# - Good for spatial data and irregular-shaped clusters.
# - Useful when clusters have arbitrary shapes.

# Local Outlier Factor (LOF):
# - Compares the local density of a point with its neighbors.
# - Points with much lower local density are considered anomalies.
# - Good when normal data has different local densities.

# Simple comparison:
#
# Isolation Forest → Random isolation
# DBSCAN          → Density + clustering
# LOF             → Local density comparison


# ============================================================
# QUESTION 3
# Key Components of a Time Series
# ============================================================

# 1. Trend:
# Long-term upward or downward movement.
# Example: Increasing yearly electricity demand.

# 2. Seasonality:
# A repeating pattern at fixed intervals.
# Example: Airline passengers increasing every summer.

# 3. Cyclical:
# Long-term fluctuations without a fixed period.
# Example: Economic expansion and recession.

# 4. Residual / Noise:
# Random variation that cannot be explained by the other components.
# Example: Unexpected increase in passengers on one particular day.


# ============================================================
# QUESTION 4
# Stationary Time Series
# ============================================================

# A stationary time series has statistical properties that remain
# approximately constant over time.

# Main properties:
# - Constant mean
# - Constant variance
# - Stable autocorrelation structure

# Testing stationarity:
# 1. Visual inspection
# 2. ADF (Augmented Dickey-Fuller) test
# 3. KPSS test

# ADF:
# p-value < 0.05 → Usually considered stationary.
# p-value >= 0.05 → Evidence of non-stationarity.

# Transforming non-stationary data:
# - Differencing
# - Log transformation
# - Square-root transformation
# - Removing trend
# - Seasonal differencing

# Example:
# Differencing:
# y_diff = y.diff()


# ============================================================
# QUESTION 5
# AR vs MA vs ARIMA vs SARIMA vs SARIMAX
# ============================================================

# AR (AutoRegressive):
# Uses previous values of the series to predict the current value.
# Example: Today's demand depends on previous days' demand.

# MA (Moving Average):
# Uses previous forecast errors to predict the current value.

# ARIMA:
# Combines:
# AR + Differencing + MA
#
# Structure:
# ARIMA(p, d, q)

# SARIMA:
# ARIMA + Seasonal components.
#
# Structure:
# SARIMA(p,d,q)(P,D,Q,s)
#
# Used when data has repeating seasonal patterns.

# SARIMAX:
# SARIMA + External/Exogenous variables.
#
# Example:
# Forecast electricity demand using historical demand + temperature.

# Summary:
#
# AR       → Previous values
# MA       → Previous errors
# ARIMA    → AR + Differencing + MA
# SARIMA   → ARIMA + Seasonality
# SARIMAX  → SARIMA + External Variables


# ============================================================
# QUESTION 6
# AirPassengers + Time Series Decomposition
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.datasets import get_rdataset
from statsmodels.tsa.seasonal import seasonal_decompose

# Load AirPassengers dataset
data = get_rdataset("AirPassengers", package="datasets").data

data["time"] = pd.to_datetime(data["time"])
data = data.set_index("time")

series = data["value"]

# Original series
plt.figure(figsize=(12, 5))
plt.plot(series)
plt.title("AirPassengers Original Series")
plt.xlabel("Year")
plt.ylabel("Passengers")
plt.show()

# Decomposition
decomposition = seasonal_decompose(
    series,
    model="multiplicative",
    period=12
)

decomposition.plot()
plt.show()


# ============================================================
# QUESTION 7
# NYC Taxi Fare + Isolation Forest
# ============================================================

# For a real NYC Taxi Fare CSV, use:
#
# df = pd.read_csv("nyc_taxi_fare.csv")
#
# Expected numerical columns can include:
# pickup_longitude
# pickup_latitude
# dropoff_longitude
# dropoff_latitude
# fare_amount
#
# Example using a numerical dataset:

from sklearn.ensemble import IsolationForest
from sklearn.datasets import load_diabetes

data = load_diabetes()

X = data.data

# Use first two numerical features for visualization
X_2d = X[:, :2]

model = IsolationForest(
    contamination=0.05,
    random_state=42
)

predictions = model.fit_predict(X)

# -1 = anomaly
#  1 = normal
anomalies = predictions == -1

plt.figure(figsize=(8, 6))

plt.scatter(
    X_2d[~anomalies, 0],
    X_2d[~anomalies, 1],
    label="Normal"
)

plt.scatter(
    X_2d[anomalies, 0],
    X_2d[anomalies, 1],
    marker="x",
    s=100,
    label="Anomaly"
)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Isolation Forest Anomaly Detection")
plt.legend()
plt.show()


# ============================================================
# QUESTION 8
# AirPassengers + SARIMA + 12-Month Forecast
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.datasets import get_rdataset
from statsmodels.tsa.statespace.sarimax import SARIMAX

data = get_rdataset("AirPassengers", package="datasets").data

data["time"] = pd.to_datetime(data["time"])
data = data.set_index("time")

series = data["value"]

# SARIMA model
model = SARIMAX(
    series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
)

result = model.fit(disp=False)

# Forecast next 12 months
forecast = result.forecast(steps=12)

print("Next 12 Months Forecast:")
print(forecast)

# Plot
plt.figure(figsize=(12, 5))

plt.plot(series, label="Actual")
plt.plot(
    forecast,
    label="Forecast",
    linestyle="--"
)

plt.title("SARIMA Forecast - AirPassengers")
plt.xlabel("Year")
plt.ylabel("Passengers")
plt.legend()
plt.show()


# ============================================================
# QUESTION 9
# Local Outlier Factor (LOF)
# ============================================================

from sklearn.neighbors import LocalOutlierFactor
from sklearn.datasets import make_blobs

X, y = make_blobs(
    n_samples=500,
    centers=3,
    cluster_std=1,
    random_state=42
)

# Add some artificial outliers
import numpy as np

outliers = np.array([
    [10, 10],
    [-10, -10],
    [15, -12],
    [-12, 15]
])

X = np.vstack([X, outliers])

model = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.01
)

predictions = model.fit_predict(X)

# -1 = anomaly
anomalies = predictions == -1

plt.figure(figsize=(8, 6))

plt.scatter(
    X[~anomalies, 0],
    X[~anomalies, 1],
    label="Normal"
)

plt.scatter(
    X[anomalies, 0],
    X[anomalies, 1],
    marker="x",
    s=100,
    label="Anomaly"
)

plt.title("Local Outlier Factor")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.legend()
plt.show()


# ============================================================
# QUESTION 10
# POWER GRID REAL-TIME DATA SCIENCE WORKFLOW
# ============================================================

# Dataset:
# timestamp
# region
# weather conditions
# energy usage


# STEP 1: Data Collection
# -----------------------
# Receive energy readings every 15 minutes using a streaming system/API.

# Example:
#
# Timestamp        Region    Temperature    Energy
# 10:00            Delhi       32°C          5000
# 10:15            Delhi       33°C          5100
# 10:30            Delhi       33°C          7000  <-- Possible anomaly


# STEP 2: Data Preprocessing
# -------------------------
# - Convert timestamp to datetime.
# - Sort data by timestamp.
# - Handle missing values.
# - Remove duplicate records.
# - Scale numerical features when required.
# - Encode categorical variables such as region.


# STEP 3: Real-Time Anomaly Detection
# -----------------------------------

# Isolation Forest:
# - Good choice for real-time monitoring.
# - Detects unusual combinations of energy usage,
#   temperature and other numerical features.
#
# Example:
#
# IsolationForest(
#     contamination=0.01,
#     random_state=42
# )


# LOF:
# - Useful when local behavior is important.
# - Compares each new observation with nearby observations.
# - Can detect local spikes/drops.
#
# Limitation:
# LOF can be less convenient for continuously changing streaming
# data because it relies on neighboring observations.


# DBSCAN:
# - Detects points outside dense regions as noise.
# - Useful for spatial/cluster-based patterns.
# - Less suitable for high-speed continuously changing streams
#   unless implemented with a suitable streaming/window approach.


# Practical choice:
# Isolation Forest → Primary real-time anomaly detector
# LOF             → Useful for local-density analysis
# DBSCAN          → Useful for spatial/cluster analysis


# STEP 4: Short-Term Forecasting
# ------------------------------

# Use SARIMAX when external variables such as temperature,
# weather and region affect energy demand.

# SARIMAX is a strong choice because:
# - Handles time dependence.
# - Handles seasonality.
# - Uses external variables.
#
# Example:
#
# SARIMAX(
#     energy_usage,
#     exog=weather_features,
#     order=(1,1,1),
#     seasonal_order=(1,1,1,96)
# )
#
# 96 = 96 fifteen-minute intervals in one day.


# Model selection:
#
# ARIMA:
# No seasonality/external variables.
#
# SARIMA:
# Seasonality exists.
#
# SARIMAX:
# Seasonality + external variables.
#
# For this power-grid problem:
# SARIMAX is generally the best choice.


# STEP 5: Forecast + Anomaly Detection Together
# ----------------------------------------------

# Forecast expected energy demand.

# Example:
#
# Actual usage    = 9000
# Forecast usage  = 6000
#
# Large difference → Possible anomaly.

# Therefore, use both:
#
# Forecast model → Expected demand
# Anomaly model  → Detect abnormal behavior


# STEP 6: Model Validation
# ------------------------

# Use time-based train/test splitting.
# Do NOT randomly shuffle time-series data.

# Forecast metrics:
# - MAE
# - RMSE
# - MAPE

# Example:
#
# MAE  → Average absolute forecasting error
# RMSE → Penalizes large errors
# MAPE → Percentage error


# Anomaly detection metrics:
# - Precision
# - Recall
# - F1-score
# - False Positive Rate

# If labeled anomalies are unavailable:
# - Manual investigation
# - Alert confirmation
# - Historical incident comparison


# STEP 7: Continuous Monitoring
# ----------------------------

# Monitor:
# - Forecast error
# - Number of anomalies
# - False alarms
# - Data quality
# - Model drift
# - Prediction latency

# Retrain the model when:
# - Data distribution changes.
# - Forecast performance decreases.
# - New seasonal patterns appear.


# STEP 8: Business Benefits
# -------------------------

# The system can help the power-grid company:

# - Detect abnormal electricity consumption quickly.
# - Identify equipment problems.
# - Detect sudden spikes or drops.
# - Forecast future energy demand.
# - Improve power generation planning.
# - Prevent overloads.
# - Reduce operational costs.
# - Improve grid reliability.
# - Send alerts to operators before serious failures occur.


# FINAL WORKFLOW
#
# Streaming Data
#       ↓
# Data Cleaning
#       ↓
# Feature Engineering
#       ↓
# ┌─────────────────────┐
# │                     │
# ↓                     ↓
# Anomaly Detection     Forecasting
# Isolation Forest      SARIMAX
# LOF / DBSCAN          + Weather
# │                     │
# └──────────┬──────────┘
#            ↓
#       Alerts + Forecast
#            ↓
#      Monitor Performance
#            ↓
#      Retrain / Update